In [1]:
import pandas as pd
import mysql.connector
from pathlib import Path

file_path = "C:/skn29/PYTHON/project/data/자동차보험_계약정보_전체_2023_2025.xlsx"

df = pd.read_excel(file_path)

df = df[
    [
        'isuCmpyOfrYm',
        'isuItmsNm',
        'mogClsfNm',
        'sexNm',
        'aggr',
        'atmbPlorNm',
        'kncrNm',
        'joinCnt',
        'elpsInpm'
    ]
].copy()

df = df.rename(columns={
    'isuCmpyOfrYm': 'stat_year_month',
    'isuItmsNm': 'insurance_product_name',
    'mogClsfNm': 'coverage_name',
    'sexNm': 'gender_name',
    'aggr': 'age_group',
    'atmbPlorNm': 'origin_type',
    'kncrNm': 'vehicle_class',
    'joinCnt': 'join_count',
    'elpsInpm': 'earned_premium_amount'
})

str_cols = [
    'stat_year_month',
    'insurance_product_name',
    'coverage_name',
    'gender_name',
    'age_group',
    'origin_type',
    'vehicle_class'
]

for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

gender_map = {
    '남자': 'M',
    '여자': 'F'
}
df['gender_code'] = df['gender_name'].map(gender_map)

df['stat_year_month'] = pd.to_numeric(df['stat_year_month'], errors='coerce')
df['join_count'] = pd.to_numeric(df['join_count'], errors='coerce')
df['earned_premium_amount'] = pd.to_numeric(df['earned_premium_amount'], errors='coerce')

df = df.dropna(subset=[
    'stat_year_month',
    'insurance_product_name',
    'coverage_name',
    'gender_code',
    'age_group',
    'origin_type',
    'vehicle_class',
    'join_count',
    'earned_premium_amount'
])

df['stat_year_month'] = df['stat_year_month'].astype(int).astype(str)
df['join_count'] = df['join_count'].astype(int)
df['earned_premium_amount'] = df['earned_premium_amount'].astype(int)

allowed_products = ['개인용', '업무용', '영업용']
allowed_coverages = ['대인배상1', '대인배상2', '자기신체사고', '대물배상', '자기차량손해']
allowed_age_groups = ['20대 이하', '30대', '40대', '50대', '60대', '70대 이상']
allowed_origin_types = ['국산', '외산']
allowed_vehicle_classes = ['소형', '중형', '대형', '다인승']

df = df[df['insurance_product_name'].isin(allowed_products)].copy()
df = df[df['coverage_name'].isin(allowed_coverages)].copy()
df = df[df['age_group'].isin(allowed_age_groups)].copy()
df = df[df['origin_type'].isin(allowed_origin_types)].copy()
df = df[df['vehicle_class'].isin(allowed_vehicle_classes)].copy()
df = df[df['join_count'] >= 0].copy()

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='!didwjdgus16',
    database='car_insurance_final',
    charset='utf8mb4'
)

cursor = conn.cursor(dictionary=True)

cursor.execute("SELECT insurance_product_id, insurance_product_name FROM insurance_product")
product_map = {row['insurance_product_name']: row['insurance_product_id'] for row in cursor.fetchall()}

cursor.execute("SELECT coverage_id, coverage_name FROM coverage_master")
coverage_map = {row['coverage_name']: row['coverage_id'] for row in cursor.fetchall()}

df['insurance_product_id'] = df['insurance_product_name'].map(product_map)
df['coverage_id'] = df['coverage_name'].map(coverage_map)

df = df.dropna(subset=['insurance_product_id', 'coverage_id']).copy()
df['insurance_product_id'] = df['insurance_product_id'].astype(int)
df['coverage_id'] = df['coverage_id'].astype(int)

df = df[
    [
        'stat_year_month',
        'insurance_product_id',
        'coverage_id',
        'gender_code',
        'age_group',
        'origin_type',
        'vehicle_class',
        'join_count',
        'earned_premium_amount'
    ]
].copy()

df = df.drop_duplicates(subset=[
    'stat_year_month',
    'insurance_product_id',
    'coverage_id',
    'gender_code',
    'age_group',
    'origin_type',
    'vehicle_class'
])

print("최종 적재 대상 행 수:", len(df))
print(df['vehicle_class'].value_counts())
print(df['origin_type'].value_counts())
print(df['age_group'].value_counts())
print(df['gender_code'].value_counts())

insert_sql = """
INSERT INTO insurance_contract_stat (
    stat_year_month,
    insurance_product_id,
    coverage_id,
    gender_code,
    age_group,
    origin_type,
    vehicle_class,
    join_count,
    earned_premium_amount
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    join_count = VALUES(join_count),
    earned_premium_amount = VALUES(earned_premium_amount),
    updated_at = CURRENT_TIMESTAMP
"""

data_to_insert = [
    (
        row.stat_year_month,
        row.insurance_product_id,
        row.coverage_id,
        row.gender_code,
        row.age_group,
        row.origin_type,
        row.vehicle_class,
        row.join_count,
        row.earned_premium_amount
    )
    for row in df.itertuples(index=False)
]

cursor.close()
cursor = conn.cursor()
cursor.executemany(insert_sql, data_to_insert)
conn.commit()

print(f"{cursor.rowcount}건 처리 완료")

cursor.close()
conn.close()

최종 적재 대상 행 수: 16800
vehicle_class
대형     4200
다인승    4200
소형     4200
중형     4200
Name: count, dtype: int64
origin_type
국산    8400
외산    8400
Name: count, dtype: int64
age_group
50대       2800
20대 이하    2800
40대       2800
30대       2800
70대 이상    2800
60대       2800
Name: count, dtype: int64
gender_code
M    8400
F    8400
Name: count, dtype: int64
16800건 처리 완료
